## 1. Imports

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from datetime import datetime
from pathlib import Path
from src.paths import DATA_PROCESSED
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import csr_matrix, hstack
from sklearn.feature_extraction.text import CountVectorizer

## 2. Chargement des données

In [4]:
df = pd.read_csv(DATA_PROCESSED / "movies_preprocessed_clean.csv", index_col=0)
cast = pd.read_csv(DATA_PROCESSED / "cast_group_clean.csv", index_col=0)

Je vais merge les 2 DF voir si beaucoup de NA etc

In [5]:
df_cast = pd.merge(df, cast, how='inner', on='tconst')

In [6]:
print(f'df.shape {df.shape}')
print(f'df.cast.shape {cast.shape}')
print(f'df_cast.shape {df_cast.shape}')

df.shape (330702, 33)
df.cast.shape (682132, 2)
df_cast.shape (328561, 35)


On ne perd que 2000 films par rapport au df standard, négligeable comparé à l'importance de la feature,
on va donc essayer de produire une matrice pour entrainer un modele de CountVectorizer

Je vais crée une liste de tout les genre avant tout 

In [7]:
genre_cols = df_cast.loc[:, 'Action':'Western'].columns.tolist()

Et j'instancie le CV

In [8]:
CV = CountVectorizer()

cast_sparse = CV.fit_transform(df_cast['clean_name'])
genre_sparse = csr_matrix(df_cast[genre_cols].values)

J'ai vu que hstack permettait de coller 2 matrices, et tocsr le convertit dans un format qui permet de découper une seule ligne

In [9]:
matrice = hstack([cast_sparse, genre_sparse]).tocsr()

In [10]:
position = 0
scores = cosine_similarity(matrice[position], matrice)

scores = scores.flatten()
ordre = scores.argsort()[::-1]
top = ordre[1:11]

J'englobe tout ca dans une fonction

In [11]:
def recherche_par_titre(titre, n=10):
    matches = df_cast[df_cast['primaryTitle'].str.contains(titre, case=False, na=False)]
    if matches.empty:
        return f"Aucun film trouvé pour '{titre}'"
    position = matches.sort_values('weight_rating', ascending=False).index[0]
    
    sims = cosine_similarity(matrice[position], matrice).flatten()
    ordre = sims.argsort()[::-1]
    top = ordre[1:n+1]
    return df_cast.iloc[top][['primaryTitle', 'startYear', 'clean_name', 'weight_rating']]


In [12]:
recherche_par_titre('matrix')

,primaryTitle,startYear,clean_name,weight_rating
95490,The Matrix Revolutions,2003.0,keanureeves laurencefishburne carrie-annemoss ...,6.694208
93852,The Matrix Reloaded,2003.0,keanureeves laurencefishburne carrie-annemoss ...,7.190439
150744,The Matrix Resurrections,2021.0,keanureeves keanureeves carrie-annemoss carrie...,5.611176
275420,Iashmiir Official Version,2014.0,albertogreco,6.153038
317814,Greenpool,2018.0,jacobjean,6.162981
234196,The World in 2080,2023.0,rickyriyaf,6.160502
80062,New Blood,1999.0,johnhurt nickmoran carrie-annemoss shawnwayans...,6.053620
316207,Kung Fu Traveler 2,2017.0,huchen,6.119657
90536,UFO: Distruggete Base Luna,1971.0,davidlane kenturner,6.162299
19096,Lost Planet Airmen,1951.0,fredcbrannon,6.139437


On a une part de pertinence, mais une fois cette pertinence épuisée le résultat s'effondre : on part dans des films complètement inconnus qui ne partagent qu'un genre. 